# 010-Vectorization of Statement (문장의 vector 화)

- BOW (Bag of Words) 
- TF-IDF (Term Frequency - Inverse Document Frequency)  
- Word Embedding - Keras word API 사용

In [2]:
import pandas as pd

sentences = ['I love my dog.', 
             'I love my cat.', 
             'You love my dog!',
             'Do you think my dog is amazing?']

## 1. Bag of Word (BOW)

In [3]:
from sklearn.feature_extraction.text import CountVectorizer

count_vectorizer = CountVectorizer()
count_vectorizer

CountVectorizer(analyzer='word', binary=False, decode_error='strict',
                dtype=<class 'numpy.int64'>, encoding='utf-8', input='content',
                lowercase=True, max_df=1.0, max_features=None, min_df=1,
                ngram_range=(1, 1), preprocessor=None, stop_words=None,
                strip_accents=None, token_pattern='(?u)\\b\\w\\w+\\b',
                tokenizer=None, vocabulary=None)

In [4]:
features = count_vectorizer.fit_transform(sentences)
features

<4x9 sparse matrix of type '<class 'numpy.int64'>'
	with 17 stored elements in Compressed Sparse Row format>

In [5]:
vectorized_sentences = features.toarray()
vectorized_sentences

array([[0, 0, 0, 1, 0, 1, 1, 0, 0],
       [0, 1, 0, 0, 0, 1, 1, 0, 0],
       [0, 0, 0, 1, 0, 1, 1, 0, 1],
       [1, 0, 1, 1, 1, 0, 1, 1, 1]], dtype=int64)

### features 의 단어 list

In [6]:
feature_names = count_vectorizer.get_feature_names()
feature_names

['amazing', 'cat', 'do', 'dog', 'is', 'love', 'my', 'think', 'you']

In [7]:
df = pd.DataFrame(vectorized_sentences, columns=feature_names)
df.index.name = 'sentence'
df

,amazing,cat,do,dog,is,love,my,think,you
sentence,,,,,,,,,
0,0,0,0,1,0,1,1,0,0
1,0,1,0,0,0,1,1,0,0
2,0,0,0,1,0,1,1,0,1
3,1,0,1,1,1,0,1,1,1


In [8]:
sentences

['I love my dog.',
 'I love my cat.',
 'You love my dog!',
 'Do you think my dog is amazing?']

## 2. TF-IDF

- TF-IDF(Term Frequency - Inverse Document Frequency)

In [9]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf_vectorizer = TfidfVectorizer()
tfidf_vectorizer

TfidfVectorizer(analyzer='word', binary=False, decode_error='strict',
                dtype=<class 'numpy.float64'>, encoding='utf-8',
                input='content', lowercase=True, max_df=1.0, max_features=None,
                min_df=1, ngram_range=(1, 1), norm='l2', preprocessor=None,
                smooth_idf=True, stop_words=None, strip_accents=None,
                sublinear_tf=False, token_pattern='(?u)\\b\\w\\w+\\b',
                tokenizer=None, use_idf=True, vocabulary=None)

In [10]:
tfidf_sentences = tfidf_vectorizer.fit_transform(sentences)
tfidf_sentences

<4x9 sparse matrix of type '<class 'numpy.float64'>'
	with 17 stored elements in Compressed Sparse Row format>

In [11]:
tfidf_vect_sentences = tfidf_sentences.toarray()
print(tfidf_vect_sentences)

[[0.         0.         0.         0.61217198 0.         0.61217198
  0.5004907  0.         0.        ]
 [0.         0.77157901 0.         0.         0.         0.49248889
  0.40264194 0.         0.        ]
 [0.         0.         0.         0.48829139 0.         0.48829139
  0.39921021 0.         0.60313701]
 [0.4343181  0.         0.4343181  0.27721962 0.4343181  0.
  0.2266452  0.4343181  0.34242138]]


In [12]:
tfidf_feature_names = tfidf_vectorizer.get_feature_names()
tfidf_feature_names

['amazing', 'cat', 'do', 'dog', 'is', 'love', 'my', 'think', 'you']

In [13]:
df = pd.DataFrame(tfidf_vect_sentences, columns=tfidf_feature_names)
df.index.name = 'sentence'
df

,amazing,cat,do,dog,is,love,my,think,you
sentence,,,,,,,,,
0,0.000000,0.000000,0.000000,0.612172,0.000000,0.612172,0.500491,0.000000,0.000000
1,0.000000,0.771579,0.000000,0.000000,0.000000,0.492489,0.402642,0.000000,0.000000
2,0.000000,0.000000,0.000000,0.488291,0.000000,0.488291,0.399210,0.000000,0.603137
3,0.434318,0.000000,0.434318,0.277220,0.434318,0.000000,0.226645,0.434318,0.342421


In [14]:
sentences

['I love my dog.',
 'I love my cat.',
 'You love my dog!',
 'Do you think my dog is amazing?']

# 3. keras word encoding

- keras  API 이용

In [15]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.utils import to_categorical

## Tokenize

In [16]:
tokenizer = Tokenizer(num_words=100, oov_token='<OOV>')    # take top 100 words from the sentences

## Word Index Vocabulary 작성

In [17]:
tokenizer.fit_on_texts(sentences)
word_index = tokenizer.word_index
word_index

{'<OOV>': 1,
 'my': 2,
 'love': 3,
 'dog': 4,
 'i': 5,
 'you': 6,
 'cat': 7,
 'do': 8,
 'think': 9,
 'is': 10,
 'amazing': 11}

## text 의 sentence 변환 및 paddding

- texts_to_sequences: text list 내의 각 text 를 수열 (sequence of integers) 로 convert


    - 입력 : text (strings) list
    - 반환 : sequence list
    
- pad_sequences: 동일한 길이로 sequence 를 zero padding

In [18]:
sequences = tokenizer.texts_to_sequences(sentences)
padded = pad_sequences(sequences, padding='post', truncating='post')

In [19]:
print(sequences)
print()
print(padded)

[[5, 3, 2, 4], [5, 3, 2, 7], [6, 3, 2, 4], [8, 6, 9, 2, 4, 10, 11]]

[[ 5  3  2  4  0  0  0]
 [ 5  3  2  7  0  0  0]
 [ 6  3  2  4  0  0  0]
 [ 8  6  9  2  4 10 11]]


In [20]:
index_word = tokenizer.index_word
index_word

{1: '<OOV>',
 2: 'my',
 3: 'love',
 4: 'dog',
 5: 'i',
 6: 'you',
 7: 'cat',
 8: 'do',
 9: 'think',
 10: 'is',
 11: 'amazing'}

### sequenced sentence 를 word sentence 로 환원

In [21]:
for sequence in sequences:
    sent = []
    for idx in sequence:
        sent.append(index_word[idx])
    print(' '.join(sent))

i love my dog
i love my cat
you love my dog
do you think my dog is amazing


# validation data 의 sequence 변환 및 padding

- 반드시 train data 에 fit 한 tokenizer 이용

In [22]:
test_data = ['I really love my dog',
             'my dog loves my lizard']

test_seq = tokenizer.texts_to_sequences(test_data)
test_padded = pad_sequences(test_seq, maxlen=10, padding='post')

print(test_seq)
print(word_index)
print(test_padded)

[[5, 1, 3, 2, 4], [2, 4, 1, 2, 1]]
{'<OOV>': 1, 'my': 2, 'love': 3, 'dog': 4, 'i': 5, 'you': 6, 'cat': 7, 'do': 8, 'think': 9, 'is': 10, 'amazing': 11}
[[5 1 3 2 4 0 0 0 0 0]
 [2 4 1 2 1 0 0 0 0 0]]


## One-Hot-Encoding 표현

In [30]:
sentence = ["I really think this is amazing. honest."]
sequence = tokenizer.texts_to_sequences(sentence)
print(sequence)
print()
print(to_categorical(sequence))

[[5, 1, 9, 1, 10, 11, 1]]

[[[0. 0. 0. 0. 0. 1. 0. 0. 0. 0. 0. 0.]
  [0. 1. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
  [0. 0. 0. 0. 0. 0. 0. 0. 0. 1. 0. 0.]
  [0. 1. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
  [0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 1. 0.]
  [0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 1.]
  [0. 1. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]]]
